# Using Logistic Regression to Find the Number of Wins Needed to Make the Playoffs

In [ ]:
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px

df = pd.read_csv('data/baseball.csv')
data = df.copy()

## Remove data after 2001
data = df[ df['Year'] <= 2001].copy()

## Maybe add scatter... 
fig = px.scatter(df, x='W', y='Team', color=df['Playoffs'].map({0: 'No', 1: 'Yes'}),
                 hover_data=['Year'], labels={'color': 'Made playoffs'}, height=800)
fig.show()

In [ ]:
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px


df = pd.read_csv('data/baseball.csv')
data = df.copy()

## Remove data after 2001
data = df[ df['Year'] <= 2001].copy()

# Define the independent (Wins) and dependent (Playoff status) variables
X = sm.add_constant(data[['W']])  # Add constant for intercept
y = data['Playoffs']  # Playoff status

# Fit the logistic regression model
logit_model = sm.Logit(y, X).fit()

# Display the summary of the logistic regression model
print(logit_model.summary())

# Predict the probability of making the playoffs for a range of wins
wins_range = pd.DataFrame({'W': range(60, 110)})  # Wins from 60 to 109
wins_range = sm.add_constant(wins_range)  # Add constant

# Generate playoff probabilities using the logistic regression model
wins_range['Playoff_Probability'] = logit_model.predict(wins_range)


wins_range.iloc[30:40]

In [ ]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# Predict playoff probabilities
wins_range = pd.DataFrame({"W": range(60, 110)})
X = sm.add_constant(wins_range, has_constant="add")
wins_range["Playoff_Probability"] = logit_model.predict(X)

# Create the chart
fig = px.line(
    wins_range,
    x="W",
    y="Playoff_Probability",
    template="plotly_white",
    labels={
        "W": "Regular-season wins",
        "Playoff_Probability": "Playoff probability",
    },
    title=(
        "<b>How many wins get you into October?</b>"
        "<br><sup>Estimated MLB playoff probability by season win total</sup>"
    ),
)

fig.update_traces(
    line=dict(color="#2563EB", width=4),
    fill="tozeroy",
    fillcolor="rgba(37, 99, 235, 0.08)",
    hovertemplate="<b>%{x} wins</b><br>Playoff chance: %{y:.1%}<extra></extra>",
)

# Highlight the 50% probability threshold
fig.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="#F59E0B",
    annotation_text="50% chance",
    annotation_position="top left",
    annotation_font_color="#B45309",
)

fig.update_layout(
    height=500,
    font=dict(family="Arial, sans-serif", size=14, color="#334155"),
    title=dict(x=0.04, font=dict(size=23)),
    margin=dict(l=65, r=30, t=100, b=65),
    hovermode="x",
    hoverlabel=dict(bgcolor="white", font_size=15),
    showlegend=False,
)

fig.update_xaxes(
    dtick=5,
    showgrid=False,
    zeroline=False,
)

fig.update_yaxes(
    tickformat=".0%",
    range=[0, 1.03],
    dtick=0.2,
    gridcolor="#E2E8F0",
    zeroline=False,
)

fig.show()

In [ ]:
df = pd.read_csv('data/baseball.csv')
data = df.copy()

## Remove data after 2001
data = df[ df['Year'] <= 2001].copy()

fig = px.scatter(df, x='W', y='Team', color=df['Playoffs'].map({0: 'No', 1: 'Yes'}),
                 hover_data=['Year'], labels={'color': 'Made playoffs'}, height=800)
fig.add_vline(x=95, line_color='red')
fig.show()

# Below just confirms the model is good. 

In [ ]:
import pandas as pd
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss,
)


df = pd.read_csv('data/baseball.csv')
data = df.copy()

# 1. Define inputs and outcome
X = sm.add_constant(data[["W"]], has_constant="add")
y = data["Playoffs"].astype(int)

# 2. Split into training and test sets
# stratify keeps the playoff/non-playoff proportions similar
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# 3. Fit using ONLY the training data
logit_model = sm.Logit(y_train, X_train).fit(disp=False)

# 4. Predict on the unseen test data
test_probabilities = logit_model.predict(X_test)
test_predictions = (test_probabilities >= 0.50).astype(int)

# Baseline: always predict the most common training outcome
baseline_predictions = [y_train.mode().iloc[0]] * len(y_test)

# 5. Evaluate
metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Precision": precision_score(y_test, test_predictions, zero_division=0),
    "Recall": recall_score(y_test, test_predictions, zero_division=0),
    "F1 score": f1_score(y_test, test_predictions, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, test_probabilities),
})

print(f"Training rows: {len(y_train)} | Test rows: {len(y_test)}")
print("\nTEST METRICS")
print(metrics.round(3).to_string())

print("\nCONFUSION MATRIX")
print(pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=["Actually missed", "Actually made playoffs"],
    columns=["Predicted missed", "Predicted made playoffs"],
))

# 6. Predict the curve using the training-fitted model
wins_range = pd.DataFrame({"W": range(60, 110)})
wins_range["Playoff_Probability"] = logit_model.predict(
    sm.add_constant(wins_range[["W"]], has_constant="add")
)